In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd


# Smart Balancing Script

In [ ]:
!pip install -q albumentations opencv-python

In [ ]:
import os

DEST = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/Tea"

os.makedirs(DEST, exist_ok=True)

print(os.path.exists(DEST))

True


In [ ]:
import os
import cv2
import random
import shutil
import numpy as np
from albumentations import (
    Compose,
    HorizontalFlip,
    VerticalFlip,
    RandomRotate90,
    Rotate,
    RandomBrightnessContrast,
    ShiftScaleRotate
)

# ==========================================================
# PATH
# ==========================================================

SOURCE = "/content/drive/MyDrive/Jute_Tea_Wheat/Data/Tea_Data"
DEST   = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/Tea"

# পুরোনো Folder Delete
if os.path.exists(DEST):
    shutil.rmtree(DEST)

os.makedirs(DEST, exist_ok=True)

random.seed(42)

# ==========================================================
# Target Images Per Class
# ==========================================================

TARGET = {
    "Algal_leaf_spot":350,
    "Brown_Blight":350,
    "Gray_Blight":350,
    "Healthy":350,
    "Helopelties":350,
    "Red_Rust":350,
    "Red_Spider":400
}

# ==========================================================
# Augmentation Pipeline
# ==========================================================

transform = Compose([
    HorizontalFlip(p=0.5),
    VerticalFlip(p=0.3),
    RandomRotate90(p=0.5),
    Rotate(limit=25,p=0.5),
    ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=20,
        p=0.5
    ),
    RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    )
])

print("="*70)

total = 0

for cls in sorted(os.listdir(SOURCE)):

    class_path = os.path.join(SOURCE,cls)

    if not os.path.isdir(class_path):
        continue

    save_path = os.path.join(DEST,cls)
    os.makedirs(save_path,exist_ok=True)

    imgs = [x for x in os.listdir(class_path)
            if x.lower().endswith((".jpg",".jpeg",".png"))]

    target = TARGET[cls]

    print(f"\n{cls}")
    print("Original :",len(imgs))
    print("Target   :",target)

    # -----------------------------------------------------
    # Case 1 : Enough Images
    # -----------------------------------------------------

    if len(imgs)>=target:

        selected = random.sample(imgs,target)

        for img in selected:

            shutil.copy(
                os.path.join(class_path,img),
                os.path.join(save_path,img)
            )

    # -----------------------------------------------------
    # Case 2 : Need Augmentation
    # -----------------------------------------------------

    else:

        # Copy Original

        for img in imgs:

            shutil.copy(
                os.path.join(class_path,img),
                os.path.join(save_path,img)
            )

        current = len(imgs)

        while current < target:

            img_name = random.choice(imgs)

            img_path = os.path.join(class_path,img_name)

            image = cv2.imread(img_path)

            image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)

            aug = transform(image=image)["image"]

            save_name = f"aug_{current}_{img_name}"

            cv2.imwrite(
                os.path.join(save_path,save_name),
                cv2.cvtColor(aug,cv2.COLOR_RGB2BGR)
            )

            current += 1

    total += target

print("\n"+"="*70)
print("Finished Successfully")
print("Total Images :",total)
print("="*70)


Algal_leaf_spot
Original : 1799
Target   : 350

Brown_Blight
Original : 1957
Target   : 350

Gray_Blight
Original : 1949
Target   : 350

Healthy
Original : 1782
Target   : 350

Helopelties
Original : 1879
Target   : 350

Red_Rust
Original : 2131
Target   : 350

Red_Spider
Original : 1588
Target   : 400

Finished Successfully
Total Images : 2500


In [ ]:
import os
import pandas as pd

# ======================================================
# Dataset Path
# ======================================================
DATASET_PATH = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data"

summary = []

print("="*70)

grand_total = 0

for crop in sorted(os.listdir(DATASET_PATH)):

    crop_path = os.path.join(DATASET_PATH, crop)

    if not os.path.isdir(crop_path):
        continue

    print(f"\n📂 {crop}")

    crop_total = 0

    for disease in sorted(os.listdir(crop_path)):

        disease_path = os.path.join(crop_path, disease)

        if not os.path.isdir(disease_path):
            continue

        count = len([
            f for f in os.listdir(disease_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))
        ])

        crop_total += count

        summary.append({
            "Crop": crop,
            "Disease": disease,
            "Images": count
        })

        print(f"   {disease:<25} : {count}")

    grand_total += crop_total

    print(f"\n   Total {crop} Images : {crop_total}")

print("\n" + "="*70)
print(f"Grand Total Images : {grand_total}")
print("="*70)

# ======================================================
# Save CSV Report
# ======================================================

df = pd.DataFrame(summary)

csv_path = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/dataset_summary.csv"

df.to_csv(csv_path, index=False)

print(f"\n✅ CSV Saved Successfully!")
print(csv_path)

display(df)


📂 Jute_Data
   Cescospora_Leaf_Spot      : 309
   Golden_Mosaic             : 347
   Healthy_Leaf              : 264

   Total Jute_Data Images : 920

📂 Tea_Data
   Algal_leaf_spot           : 350
   Brown_Blight              : 350
   Gray_Blight               : 350
   Healthy                   : 350
   Helopelties               : 350
   Red_Rust                  : 350
   Red_Spider                : 400

   Total Tea_Data Images : 2500

📂 Wheat_Data
   BlackPoint                : 303
   FusariumFootRot           : 248
   HealthyLeaf               : 250
   LeafBlight                : 364
   WheatBlast                : 310

   Total Wheat_Data Images : 1475

Grand Total Images : 4895

✅ CSV Saved Successfully!
/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/dataset_summary.csv


,Crop,Disease,Images
0,Jute_Data,Cescospora_Leaf_Spot,309
1,Jute_Data,Golden_Mosaic,347
2,Jute_Data,Healthy_Leaf,264
3,Tea_Data,Algal_leaf_spot,350
4,Tea_Data,Brown_Blight,350
5,Tea_Data,Gray_Blight,350
6,Tea_Data,Healthy,350
7,Tea_Data,Helopelties,350
8,Tea_Data,Red_Rust,350
9,Tea_Data,Red_Spider,400


# Duplicate Image Check

In [ ]:
import os
import hashlib
from collections import defaultdict

DATASET = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data"

duplicates = defaultdict(list)

for root, dirs, files in os.walk(DATASET):

    for file in files:

        if file.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):

            path = os.path.join(root,file)

            with open(path,'rb') as f:
                filehash = hashlib.md5(f.read()).hexdigest()

            duplicates[filehash].append(path)

dup_list = []

for h,paths in duplicates.items():

    if len(paths)>1:

        dup_list.append(paths)

print("="*60)
print("Duplicate Groups :",len(dup_list))

total_duplicate = sum(len(x)-1 for x in dup_list)

print("Duplicate Images :",total_duplicate)

if total_duplicate==0:
    print("\n✅ No Duplicate Found")
else:
    print("\nExample Duplicate:")
    print(*dup_list[0],sep="\n")

Duplicate Groups : 0
Duplicate Images : 0

✅ No Duplicate Found


In [ ]:
# import matplotlib.pyplot as plt
# from PIL import Image

# img1 = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/Wheat_Data/FusariumFootRot/FusariumFootRot_29.jpg"
# img2 = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/Wheat_Data/FusariumFootRot/FusariumFootRot_28.jpg"

# plt.figure(figsize=(8,4))

# plt.subplot(1,2,1)
# plt.imshow(Image.open(img1))
# plt.title("Image 1")
# plt.axis("off")

# plt.subplot(1,2,2)
# plt.imshow(Image.open(img2))
# plt.title("Image 2")
# plt.axis("off")

# plt.show()

# Duplicate Remove

In [ ]:
import os
import hashlib
from collections import defaultdict

DATASET = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data"

duplicates = defaultdict(list)

for root, dirs, files in os.walk(DATASET):

    for file in files:

        if file.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):

            path = os.path.join(root,file)

            with open(path,'rb') as f:
                filehash = hashlib.md5(f.read()).hexdigest()

            duplicates[filehash].append(path)

removed = 0

for h, paths in duplicates.items():

    if len(paths) > 1:

        # প্রথমটা রাখবে
        for dup in paths[1:]:

            os.remove(dup)
            removed += 1

print("="*60)
print("Duplicate Removed :", removed)

Duplicate Removed : 128


# Corrupted Image Check

In [ ]:
from PIL import Image
import os

DATASET="/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data"

bad=[]

for root,dirs,files in os.walk(DATASET):

    for file in files:

        if file.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):

            path=os.path.join(root,file)

            try:

                img=Image.open(path)
                img.verify()

            except:

                bad.append(path)

print("="*60)
print("Corrupted Images :",len(bad))

if len(bad)==0:

    print("\n✅ No Corrupted Image")

else:

    print("\nExample:")
    print(bad[0])

Corrupted Images : 0

✅ No Corrupted Image


# CSV From Balanced_Data

In [ ]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data"

rows = []

for crop in sorted(os.listdir(ROOT)):

    crop_path = os.path.join(ROOT, crop)

    if not os.path.isdir(crop_path):
        continue

    for disease in sorted(os.listdir(crop_path)):

        disease_path = os.path.join(crop_path, disease)

        if not os.path.isdir(disease_path):
            continue

        label = crop + "_" + disease

        for img in os.listdir(disease_path):

            if img.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):

                rows.append({
                    "image_path": os.path.join(disease_path, img),
                    "crop": crop,
                    "disease": disease,
                    "label": label
                })

df = pd.DataFrame(rows)

print("Total Images :", len(df))
display(df.head())

Total Images : 4895


,image_path,crop,disease,label
0,/content/drive/MyDrive/Jute_Tea_Wheat/Balanced...,Jute_Data,Cescospora_Leaf_Spot,Jute_Data_Cescospora_Leaf_Spot
1,/content/drive/MyDrive/Jute_Tea_Wheat/Balanced...,Jute_Data,Cescospora_Leaf_Spot,Jute_Data_Cescospora_Leaf_Spot
2,/content/drive/MyDrive/Jute_Tea_Wheat/Balanced...,Jute_Data,Cescospora_Leaf_Spot,Jute_Data_Cescospora_Leaf_Spot
3,/content/drive/MyDrive/Jute_Tea_Wheat/Balanced...,Jute_Data,Cescospora_Leaf_Spot,Jute_Data_Cescospora_Leaf_Spot
4,/content/drive/MyDrive/Jute_Tea_Wheat/Balanced...,Jute_Data,Cescospora_Leaf_Spot,Jute_Data_Cescospora_Leaf_Spot


In [ ]:
import os

SAVE_PATH = "/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/dataset.csv"

df.to_csv(SAVE_PATH, index=False)

print("✅ CSV Saved Successfully!")
print(SAVE_PATH)

✅ CSV Saved Successfully!
/content/drive/MyDrive/Jute_Tea_Wheat/Balanced_Data/dataset.csv


# Stratified Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train : " , len(train_df))
print("Validation : " , len(val_df))
print("Test : " , len(test_df))

Train :  3426
Validation :  734
Test :  735


# Folder Create

In [ ]:
import os

SAVE_ROOT="/content/drive/MyDrive/Jute_Tea_Wheat/Dataset"

for split in ["train","val","test"]:

    os.makedirs(os.path.join(SAVE_ROOT,split),exist_ok=True)

# Image Copy

In [ ]:
import shutil

def copy_images(dataframe, split):

    for _,row in dataframe.iterrows():

        src=row["image_path"]

        dst=os.path.join(
            SAVE_ROOT,
            split,
            row["crop"],
            row["disease"]
        )

        os.makedirs(dst,exist_ok=True)

        shutil.copy2(src,dst)

copy_images(train_df,"train")
copy_images(val_df,"val")
copy_images(test_df,"test")

print("Dataset Split Completed")

Dataset Split Completed


# CSV Save


In [ ]:
train_df.to_csv(os.path.join(SAVE_ROOT,"train.csv"),index=False)
val_df.to_csv(os.path.join(SAVE_ROOT,"val.csv"),index=False)
test_df.to_csv(os.path.join(SAVE_ROOT,"test.csv"),index=False)

print("CSV Saved")

CSV Saved


In [ ]:
import os
import pandas as pd

#=========================
# Dataset Path
#=========================
DATASET_PATH = "/content/drive/MyDrive/Jute_Tea_Wheat/Data"

data = []

# Crop Encoding
crop_encoding = {
    "Tea_Data":0,
    "Jute_Data":1,
    "Wheat_Data":2
}

disease_id = 0
disease_encoding = {}

for crop in sorted(os.listdir(DATASET_PATH)):

    crop_path = os.path.join(DATASET_PATH, crop)

    if not os.path.isdir(crop_path):
        continue

    for disease in sorted(os.listdir(crop_path)):

        disease_path = os.path.join(crop_path, disease)

        if not os.path.isdir(disease_path):
            continue

        if disease not in disease_encoding:
            disease_encoding[disease] = disease_id
            disease_id += 1

        for img in os.listdir(disease_path):

            if img.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tif','.tiff')):

                img_path = os.path.join(crop_path, disease, img)

                data.append({
                    "image_path": img_path,
                    "crop": crop,
                    "crop_id": crop_encoding[crop],
                    "disease": disease,
                    "disease_id": disease_encoding[disease]
                })

df = pd.DataFrame(data)

csv_path = "/content/drive/MyDrive/Jute_Tea_Wheat/dataset.csv"

df.to_csv(csv_path,index=False)

print("CSV Saved Successfully!")
print(csv_path)

print("\nTotal Images :",len(df))

display(df.head())